# Churn — exploration & training

Mon premier modèle, bricolé en local. Ça marche, j'ai pas eu le temps de nettoyer.

In [ ]:
# imports + tout en vrac
import pandas as pd
import numpy as np
import os, pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score

df = pd.read_csv('../data/telco_churn.csv')
print(df.shape)
df.head()

In [ ]:
# preprocessing... TotalCharges il y a des espaces vides il parait
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df = df.drop(columns=['customerID'])

y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])

# je split direct, 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('accuracy', accuracy_score(y_test, y_pred))
print('precision', precision_score(y_test, y_pred))
print('recall', recall_score(y_test, y_pred))
print('f1', f1_score(y_test, y_pred))
print('roc_auc', roc_auc_score(y_test, y_proba))
print('---')
print(classification_report(y_test, y_pred))

In [ ]:
# bon je sauvegarde, j'enverrai le pkl par mail à Marc
os.makedirs('../models', exist_ok=True)
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('ok ->', os.path.abspath('../models/best_model.pkl'))

todo: refaire ça propre quand j'aurai le temps